# Análisis del backtesting (Parcial MCD-402)

Lee la corrida más reciente de `data/output/` y grafica, para los meses posteriores al corte (`p7`–`p10`):

1. AUC por ola del campeón, con la línea de `AUC_train` y el umbral de alerta de drift.
2. **Lift en el top 10 %** (métrica primaria de negocio) por ola, para todos los modelos.
3. PSI (concept drift) por ola del campeón.

In [ ]:
import glob, json, os
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = '../data/output'  # ejecutar desde notebooks/
carpetas = sorted(c for c in glob.glob(os.path.join(OUTPUT_DIR, '*'))
                  if os.path.exists(os.path.join(c, 'results_summary.json')))
carpeta_corrida = carpetas[-1]
print('Leyendo corrida:', carpeta_corrida)

with open(os.path.join(carpeta_corrida, 'backtesting_results.json'), encoding='utf-8') as f:
    backtesting = json.load(f)
with open(os.path.join(carpeta_corrida, 'results_summary.json'), encoding='utf-8') as f:
    resumen = json.load(f)

campeon = resumen['campeon']
with open(os.path.join(carpeta_corrida, f'{campeon}_metadata.json'), encoding='utf-8') as f:
    auc_train = json.load(f)['performance']['auc_train']
print('Campeon:', campeon, '| AUC train:', auc_train)

In [ ]:
tabla_olas = pd.DataFrame(backtesting['waves'])
tabla_olas

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(tabla_olas['partition'], tabla_olas['auc'], marker='o', linewidth=2, label=f'AUC por ola ({campeon})')
ax.axhline(y=auc_train, color='gray', linestyle='--', label=f'AUC train = {auc_train}')
umbral = backtesting['drift_alert_threshold']
ax.axhline(y=auc_train * (1 - umbral / 100), color='red', linestyle=':', label=f'Umbral de alerta ({umbral}%)')
for _, fila in tabla_olas.iterrows():
    ax.annotate(f"{fila['decay_pct']}%", (fila['partition'], fila['auc']),
                textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
ax.set_xlabel('Particion / mes futuro'); ax.set_ylabel('AUC-ROC')
ax.set_title('Backtesting walk-forward: AUC por ola (sin reentrenar)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(carpeta_corrida, 'backtesting_auc_por_ola.png'), dpi=120)
plt.show()

In [ ]:
# Metrica de negocio: lift en el top % por ola, todos los modelos
fig, ax = plt.subplots(figsize=(10, 5))
for nombre, bt in backtesting['todos_los_modelos'].items():
    olas = pd.DataFrame(bt['waves'])
    estilo = dict(linewidth=3, marker='o') if nombre == campeon else dict(linewidth=1, alpha=0.6)
    ax.plot(olas['partition'], olas['lift'], label=nombre + (' (campeon)' if nombre == campeon else ''), **estilo)
ax.set_xlabel('Particion / mes futuro'); ax.set_ylabel(f"Lift top {backtesting['top_pct_negocio']}%")
ax.set_title('Metrica primaria de negocio por ola')
ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(carpeta_corrida, 'backtesting_lift_por_ola.png'), dpi=120)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colores = ['#2ca02c' if p < 0.10 else '#ff7f0e' if p < 0.25 else '#d62728' for p in tabla_olas['psi']]
ax.bar(tabla_olas['partition'], tabla_olas['psi'], color=colores)
ax.axhline(y=0.10, color='orange', linestyle='--', linewidth=1, label='PSI = 0.10 (moderado)')
ax.axhline(y=0.25, color='red', linestyle='--', linewidth=1, label='PSI = 0.25 (significativo)')
ax.set_xlabel('Particion / mes futuro'); ax.set_ylabel('PSI')
ax.set_title(f'Concept drift por ola - {campeon}')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig(os.path.join(carpeta_corrida, 'backtesting_psi_por_ola.png'), dpi=120)
plt.show()

In [ ]:
if backtesting['drift_severo_en_todas_las_olas']:
    print('ALERTA SEVERA: el campeon supero el umbral de decay en TODAS las olas futuras.')
elif backtesting['drift_alert']:
    print('Alerta parcial: al menos una ola supero el umbral de decay, pero no todas.')
else:
    print('Sin alertas de drift: el campeon se mantiene estable en todas las olas evaluadas.')
print('Estado de la seleccion:', resumen['seleccion']['estado'])
print('Motivo:', resumen['seleccion']['motivo'])